> **Notebook-first lesson.** Run cells in order. The final activity is designed to be changed and rerun.

## Mathematical Framework

Math companions for this lesson:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Track **shapes, assumptions, objective, derivatives/updates, and the conditions under which the derivation stops matching reality**.

# Lesson 26: nn.Module and the training loop

## Goal
Move from raw tensors to reusable neural-network components.



In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED)
X_train = torch.randn(160, 2)
y_train = (X_train[:, 0] + .5 * X_train[:, 1] > 0).long()
X_val = torch.randn(60, 2)
y_val = (X_val[:, 0] + .5 * X_val[:, 1] > 0).long()
device = torch.device('cpu')
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.01)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)


In [ ]:
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)



## Canonical training loop


In [ ]:
model = MLP(2, 32, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    model.train()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



## What each line means
- `model.train()`: training behavior
- forward pass: compute predictions
- loss: measure objective
- zero gradients: clear previous accumulation
- backward: compute gradients
- step: update parameters

## Why gradients accumulate
PyTorch adds new gradients to existing `.grad` values. This is useful for some workflows but dangerous if you forget to clear them.

## Exercise
Rewrite the loop with SGD. Compare SGD, momentum and Adam using identical initial weights and data.


## Runnable activity
Run this experiment. Then change one architectural, data, or optimization choice and compare.

In [ ]:
import torch
from torch import nn
torch.manual_seed(0)
X=torch.randn(400,2); y=((X[:,0]*X[:,1])>0).long()
model=nn.Sequential(nn.Linear(2,32),nn.ReLU(),nn.Linear(32,2))
opt=torch.optim.Adam(model.parameters(),lr=.02); loss_fn=nn.CrossEntropyLoss()
for epoch in range(200):
    opt.zero_grad(); logits=model(X); loss=loss_fn(logits,y); loss.backward(); opt.step()
with torch.no_grad():
    acc=(model(X).argmax(1)==y).float().mean()
print("loss",float(loss),"accuracy",float(acc))

## Explanation checkpoint
Add a Markdown cell that explains the tensor shapes, the mechanism being tested, and what changed when you modified the experiment.